# SafeShift CT1 — Chuẩn bị pilot A/E/S
Ô chạy được chỉ kiểm fixtures và xuất prompt/schema trên CPU. Notebook đang BLOCKED cho chạy model. Chạy CPU_PREFLIGHT trước.


In [ ]:
from pathlib import Path
import os
import sys

repo = Path(os.environ.get("SAFESHIFT_CT1_REPO", ".")).resolve(strict=True)
assert (repo / "scripts/ct1_pilot.py").is_file(), "Set SAFESHIFT_CT1_REPO to the reviewed checkout"
assert sys.version_info >= (3, 11), "Python 3.11+ required for CPU preparation"
if os.name == "nt" and repo.drive.upper() != "D:":
    raise ValueError("SafeShift workspace must be on D:")
workspace = Path(os.environ.get("SAFESHIFT_CT1_OUTPUT_ROOT", repo / "data/processed")).resolve()
if os.name == "nt" and workspace.drive.upper() != "D:":
    raise ValueError("SafeShift output must be on D:")
workspace.mkdir(parents=True, exist_ok=True)
offline_env = {**os.environ, "TMPDIR": str(workspace), "TEMP": str(workspace),
               "TMP": str(workspace), "PYTHONDONTWRITEBYTECODE": "1", "HF_HUB_OFFLINE": "1",
               "TRANSFORMERS_OFFLINE": "1", "CUDA_VISIBLE_DEVICES": ""}


In [ ]:
import subprocess

subprocess.run(
    [sys.executable, "-B", "-m", "unittest", "discover", "-s", "tests", "-p", "test_ct1_pilot.py", "-v"],
    cwd=repo, env=offline_env, check=True,
)


In [ ]:
bundle = workspace / "ct1-notebook-prep"
subprocess.run(
    [sys.executable, "-B", "-m", "scripts.ct1_pilot", "prepare",
     "--with-control", "--output", str(bundle)],
    cwd=repo, env=offline_env, check=True,
)
print(bundle / "receipt.json")


## Trước pilot
Theo docs/research/THESIS_CT1_EVIDENCE_PILOT.md. Khóa cohort TRAIN development, miền, nhóm nguồn và quyền ở bộ chấm riêng. Image request gồm ảnh dưới định danh mù và prompt cố định. Giữ GT/annotation ngoài môi trường runner. Native release cần notebook/runner đã duyệt theo đúng source, quyền kỹ thuật và grant pilot riêng.
